In [ ]:
# =====================================================================
from pathlib import Path
import itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

warnings.filterwarnings("ignore")
BASE_DIR = next(
    (ruta for ruta in (Path.cwd(), *Path.cwd().parents)
     if (ruta / "base de datos" / "Datos_EMBI.xlsx").is_file()),
    None,
)
if BASE_DIR is None:
    raise FileNotFoundError("No se encontró base de datos/Datos_EMBI.xlsx en la carpeta del proyecto ni en sus carpetas superiores.")
ARCHIVO = BASE_DIR / "base de datos" / "Datos_EMBI.xlsx"
DIA_CORTE = 1                              # día del mes de mar/jun/sep/dic
ORDEN = (0, 1, 2)                          # modelo elegido: ARIMA(0,1,2) (menor AIC)
SALIDA = BASE_DIR / "code" / "resultados_embi"   # misma carpeta de figuras que el script
SALIDA.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------
raw = pd.read_excel(ARCHIVO, sheet_name="Diarias", header=1)
col_fecha, col_spread = raw.columns[:2]
raw[col_fecha] = pd.to_datetime(raw[col_fecha], errors="coerce")
spread = pd.to_numeric(raw[col_spread], errors="coerce")
validas = raw[col_fecha].notna() & spread.notna()
if not validas.any():
    raise ValueError("No se encontraron filas diarias con fecha y spread válidos.")
if not validas.all():
    print(f"Filas omitidas por fecha o spread inválido/vacío: {(~validas).sum()}")
diaria = pd.Series(spread.loc[validas].to_numpy(),
                   index=pd.DatetimeIndex(raw.loc[validas, col_fecha]),
                   name="spread").sort_index()
cortes, valores = [], []
for anio in range(diaria.index.min().year, diaria.index.max().year + 1):
    for mes in (3, 6, 9, 12):
        corte = pd.Timestamp(anio, mes, DIA_CORTE)
        if diaria.index.min() <= corte <= diaria.index.max():
            cortes.append(corte)
            valores.append(diaria.loc[:corte].iloc[-1])
fechas = pd.DatetimeIndex(cortes)
y = pd.Series(valores, index=fechas, name="spread")
ly = np.log(y)                         # ln(spread)
dly = ly.diff().dropna()               # Δln(spread)
dy = y.diff().dropna()                 # Δ spread
print(f"Diaria: {diaria.index.min():%d/%m/%Y} a {diaria.index.max():%d/%m/%Y}")
print(f"Trimestral (corte día {DIA_CORTE}): {len(y)} obs., {fechas[0]:%d/%m/%Y} a {fechas[-1]:%d/%m/%Y}")
print(f"Fecha convertida a {raw[col_fecha].dtype}; spread original: {raw[col_spread].dtype}")

In [ ]:
# ---------------------------------------------------------------------
fig, ax = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
ax[0].plot(fechas, y, color="#1F4E78", lw=1.8)
ax[0].set_title("ANTES: Spread EMBIG Colombia (pbs), trimestral"); ax[0].set_ylabel("pbs")
ax[1].plot(fechas, ly, color="#2E7D32", lw=1.8)
ax[1].set_title("Transformación 1: ln(spread)  (estabiliza la varianza)"); ax[1].set_ylabel("ln")
ax[2].plot(fechas[1:], dly, color="#C0392B", lw=1.5); ax[2].axhline(0, color="gray", lw=.8)
ax[2].set_title("DESPUÉS: Δln(spread)  (estabiliza la media; serie estacionaria)"); ax[2].set_ylabel("Δln")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(SALIDA / "fig1_transformacion.png", dpi=150)

# Varianza constante: Levene (mitades) y ARCH-LM sobre residuos AR(1)
def arch_ar1(s, lags):
    s = np.asarray(s)
    res = sm.OLS(s[1:], sm.add_constant(s[:-1])).fit().resid
    return het_arch(res, nlags=lags)[1]

print("\n--- Varianza constante, antes y después ---")
for nombre, s in [("Nivel", y), ("Δ nivel", dy), ("ln", ly), ("Δln", dly)]:
    s = s.reset_index(drop=True); h = len(s) // 2
    print(f"{nombre:8s} var1={s[:h].var():.4f} var2={s[h:].var():.4f} "
          f"Levene p={stats.levene(s[:h], s[h:]).pvalue:.4f} | "
          f"ARCH(1) p={arch_ar1(s,1):.4f} ARCH(4) p={arch_ar1(s,4):.4f}")

In [ ]:
# ---------------------------------------------------------------------
# BOX-JENKINS
# Etapa 1. IDENTIFICACIÓN: raíz unitaria y FAC/FACP

In [ ]:
# ---------------------------------------------------------------------
print("\n=== ETAPA 1: IDENTIFICACIÓN ===")
def adf(s, reg, nombre):
    r = adfuller(s, regression=reg, autolag="AIC")
    print(f"ADF  {nombre:22s} est={r[0]:.3f} p={r[1]:.4f} rezagos={r[2]} crit5%={r[4]['5%']:.3f}")
def kp(s, reg, nombre):
    r = kpss(s, regression=reg, nlags=int(4 * (len(s) / 100) ** 0.25))
    print(f"KPSS {nombre:22s} est={r[0]:.3f} p={r[1]:.4f} crit5%={r[3]['5%']}")
adf(ly, "c", "ln (constante)");        kp(ly, "c", "ln (nivel)")
adf(ly, "ct", "ln (const.+tend.)");    kp(ly, "ct", "ln (tendencia)")
adf(dly, "c", "Δln (constante)");      kp(dly, "c", "Δln (nivel)")

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
plot_acf(ly, lags=24, ax=ax[0, 0], title="FAC – ln(spread)")
plot_pacf(ly, lags=24, ax=ax[1, 0], method="ywm", title="FACP – ln(spread)")
plot_acf(dly, lags=24, ax=ax[0, 1], title="FAC – Δln(spread)")
plot_pacf(dly, lags=24, ax=ax[1, 1], method="ywm", title="FACP – Δln(spread)")
plt.tight_layout(); plt.savefig(SALIDA / "fig2_FAC_FACP.png", dpi=150)

In [ ]:
# ---------------------------------------------------------------------
# Etapa 2. ESTIMACIÓN: comparación por AIC y modelo elegido ARIMA(0,1,2)

In [ ]:
# ---------------------------------------------------------------------
print("\n=== ETAPA 2: ESTIMACIÓN ===")
tabla = []
for p, q in itertools.product(range(4), range(4)):
    try:
        mod = ARIMA(ly, order=(p, 1, q)).fit()
        tabla.append((f"ARIMA({p},1,{q})", mod.aic, mod.bic))
    except Exception:
        pass
tabla = pd.DataFrame(tabla, columns=["Modelo", "AIC", "BIC"]).sort_values("AIC")
print("Comparación (referencia, d=1, sin constante):")
print(tabla.head(6).round(2).to_string(index=False))

modelo = ARIMA(ly, order=ORDEN).fit()
print(f"\nModelo elegido: ARIMA{ORDEN}")
print(modelo.summary().tables[1])
print(f"AIC={modelo.aic:.2f}  BIC={modelo.bic:.2f}")
print("Módulo raíz AR:", np.abs(modelo.arroots).round(3), " raíz MA:", np.abs(modelo.maroots).round(3))

In [ ]:
# ---------------------------------------------------------------------
# Etapa 3. DIAGNÓSTICO de residuos (se descarta el primero por la diferenciación)

In [ ]:
# ---------------------------------------------------------------------
print("\n=== ETAPA 3: DIAGNÓSTICO ===")
res = modelo.resid[1:]
lb = acorr_ljungbox(res, lags=[4, 8, 12], model_df=ORDEN[0] + ORDEN[2])
print("Ljung-Box p (4,8,12):", lb["lb_pvalue"].round(3).tolist())
print("ARCH-LM(4) p:", round(het_arch(res, nlags=4)[1], 3))
jb = stats.jarque_bera(res)
print(f"Jarque-Bera p: {jb[1]:.2e} | asimetría={stats.skew(res):.2f} curtosis={stats.kurtosis(res, fisher=False):.2f}")

fig, ax = plt.subplots(2, 1, figsize=(10, 7))
plot_acf(res, lags=24, ax=ax[0], title=f"FAC – residuos ARIMA{ORDEN}")
plot_pacf(res, lags=24, ax=ax[1], method="ywm", title=f"FACP – residuos ARIMA{ORDEN}")
plt.tight_layout(); plt.savefig(SALIDA / "fig3_residuos.png", dpi=150)

In [ ]:
# ---------------------------------------------------------------------
# Etapa 4. PRONÓSTICO a 2 trimestres (IC 95% y 90%), en pbs

In [ ]:
# ---------------------------------------------------------------------
print("\n=== ETAPA 4: PRONÓSTICO ===")
fc = modelo.get_forecast(2)
punto = np.exp(fc.predicted_mean.values)
ic95 = np.exp(fc.conf_int(alpha=0.05).values)
ic90 = np.exp(fc.conf_int(alpha=0.10).values)
fut = [fechas[-1] + pd.DateOffset(months=3), fechas[-1] + pd.DateOffset(months=6)]
pron = pd.DataFrame({"Fecha": [f"{f:%d/%m/%Y}" for f in fut], "Punto": punto,
                     "IC95_inf": ic95[:, 0], "IC95_sup": ic95[:, 1],
                     "IC90_inf": ic90[:, 0], "IC90_sup": ic90[:, 1]})
print(pron.round(1).to_string(index=False))
print(f"Último dato observado ({fechas[-1]:%d/%m/%Y}): {y.iloc[-1]:.0f} pbs")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(fechas[-24:], y[-24:], marker="o", ms=3, color="#1F4E78", label="Observado")
ax.plot([fechas[-1], *fut], [y.iloc[-1], *punto], "--o", color="#C0392B", label="Pronóstico")
ax.fill_between(fut, ic95[:, 0], ic95[:, 1], color="#C0392B", alpha=.15, label="IC 95%")
ax.fill_between(fut, ic90[:, 0], ic90[:, 1], color="#C0392B", alpha=.30, label="IC 90%")
for fecha, valor in zip(fut, punto):
    ax.annotate(f"{valor:.1f}", xy=(fecha, valor), xytext=(0, 4),
                textcoords="offset points", ha="center", va="bottom",
                color="#9C2F25", fontsize=7, fontweight="bold", zorder=10)
ax.set_title(f"Pronóstico Spread EMBIG Colombia – ARIMA{ORDEN} sobre ln"); ax.set_ylabel("pbs")
ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(SALIDA / "fig4_pronostico.png", dpi=150)
plt.show()